# Load price data

In [ ]:
import os 
os.chdir("c:\\Users\\vynde\\Documents\\GitHub\\Algotrader")
os.getcwd()

In [ ]:
import numpy as np
from numba import njit
import plotly.graph_objects as go
import pandas as pd
import pandas_ta as ta
import vectorbt as vbt
from trader.trader import Trader

In [ ]:
from trader.trader import Trader
trader = Trader()
trader.add_broker("mt5")

trader.add_data("EURUSD", start="2024-06-01", end="2024-06-30", granularities="tick")
df = trader.data[0].df
df.index = df.index.tz_convert("UTC")
df

In [ ]:
import pandas as pd
df = pd.read_csv("EURUSD_2024_1min.csv")
df["datetime"] = pd.to_datetime(df["datetime"])
df.set_index("datetime", inplace=True)
df.index = df.index.tz_convert("UTC")

In [ ]:
df

# 3 - 5 Candle Drop and Pop

In [ ]:
def streak_count(cond):
    """Count the number of consecutive True values"""
    group_id = cond.astype(int).diff().fillna(0).eq(-1).cumsum()
    # .astype(int) -> convert True/False to 1/0
    # .diff() -> find changes {-1: True->False, 1: False->True, 0: no change}
    # .fillna(0) -> first value
    # .eq(-1) -> find where the change is -1 (True->False)
    # .cumsum() -> results in a 'group id' for each consecutive streak
    return cond.groupby(group_id).transform("cumsum")
    # .groupby(group_id) -> group by the group id

def consecutive_returns(returns):
    
    # group id for consecutive returns
    # new id every time return changes between >0 <0 or 0
    values = returns.copy()
    values[values == 0] = 0
    values[values > 0] = 1
    values[values < 0] = -1
    cond = values.diff()
    cond[cond != 0] = 1
    group_id = cond.cumsum()
    return returns.groupby(group_id).transform("cumsum")


![alt text](https://i.imgur.com/BD7CeaI.png)

In [ ]:
import pandas as pd
df = pd.read_csv("EURUSD_2024_1min.csv")
df["datetime"] = pd.to_datetime(df["datetime"])
df.set_index("datetime", inplace=True)

df = df.resample("15min").agg({"open": "first", "high": "max", "low": "min", "close": "last"})

df["up"] = df["close"] > df["open"]  # green candle
df["down"] = df["close"] < df["open"]  # red candle

df["lower_highs"] = streak_count(df.high < df.high.shift(1))
df["lower_lows"] = streak_count(df.low < df.low.shift(1))
df["higher_highs"] = streak_count(df.high > df.high.shift(1))
df["higher_lows"] = streak_count(df.low > df.low.shift(1))
df["upcount"] = streak_count(df.up)
df["downcount"] = streak_count(df.down)

df["return"] = df["close"].diff().round(5)
df["consec_returns"] = consecutive_returns(df["return"].round(5)).round(5)

state_names = {
    0: "WaitForStartOfUpMove",
    1: "WaitForEndOfUpMove",  # min 3 green candles
    2: "WaitForEndOfDownMove",  # min 3 red candles / 60% max retracement
    3: "Entry"}  # green candle

def get_state(state_id):
    return state_names[state_id]

indices = []
states = []
upmoves = []
downmoves = []
state_id = 0
upmove = None
downmove = None
for i in range(len(df)):
    if get_state(state_id) == "WaitForStartOfUpMove":
       if (df["upcount"].iloc[i] >= 1):# and (df["higher_highs"].iloc[i] >= 1) and (df["higher_lows"].iloc[i] >= 1):
            up_start = df["low"].iloc[i-1:i+1].min()
            smallest_green_candle = df["close"].iloc[i] - df["open"].iloc[i]
            print(i, smallest_green_candle) if i == 4140 else None
            state_id += 1

    elif get_state(state_id) == "WaitForEndOfUpMove":
        if df["up"].iloc[i]:
            smallest_green_candle = min(smallest_green_candle, df["close"].iloc[i] - df["open"].iloc[i])
            print(i, smallest_green_candle) if (i >= 4140 and i <4150) else None
        # min 3 consecutive green candles with higher highs and higher lows
        if (df["upcount"].iloc[i-1] >= 3) and (df["higher_highs"].iloc[i-1] >= 3) and (df["higher_lows"].iloc[i-1] >= 3) and (df["down"].iloc[i]):
            up_end = df["high"].iloc[i-1:i+1].max()
            largest_red_candle = abs(df["close"].iloc[i] - df["open"].iloc[i])
            print(i, "largest red", largest_red_candle) if (i >= 4140 and i <4150) else None
            state_id += 1
        elif (df["down"].iloc[i]):
            state_id = 0

    elif get_state(state_id) == "WaitForEndOfDownMove":
        if df["down"].iloc[i]:
            largest_red_candle = max(largest_red_candle, abs(df["close"].iloc[i] - df["open"].iloc[i]))
            print(i, "largest red", largest_red_candle) if (i >= 4140 and i <4150) else None
        # min 3 consecutive red candles with lower highs and lower lows
        if (df["downcount"].iloc[i-1] >= 3) and (df["lower_highs"].iloc[i-1] >= 3) and (df["lower_lows"].iloc[i-1] >= 3) and (df["up"].iloc[i]):
            down_end = df["low"].iloc[i-1:i+1].min()
            # Check retracement
            upmove = up_end - up_start
            downmove = abs(down_end - up_end)
            if ((downmove / upmove) < 0.6) and (largest_red_candle < smallest_green_candle):
                state_id += 1
                print(largest_red_candle, smallest_green_candle)
            else:
                state_id = 0
        elif df["up"].iloc[i]:
            state_id = 0
    
    elif get_state(state_id) == "Entry":
        state_id = 0

    indices.append(i)
    upmoves.append(upmove)
    downmoves.append(downmove)
    states.append(state_id)
df["state"] = states
df["upmove"] = upmoves
df["downmove"] = downmoves
df["indices"] = indices

entries = (df[df.state == max(state_names.keys())])

# This was the vectorized approach. Too complicated to get it working
#numred = 4
#entries = df[
#    # first green candle
#    (df.upcount==1) &
#    # 3 red candles before with lower high and lower low
#    df.shift(1).eval(f"downcount=={numred}") &# and lh>={numred} and ll>={numred}") &
#    # max 60% retracement from last green candles
#    (df["consec_returns"].shift(1).abs() < 0.6 * df["consec_returns"].shift(numred+1).abs()) &
#    # 3 green candles 
#    df.shift(numred+1).eval(f"upcount>={numred}")
#    ]
#
entries[["indices", "open", "high", "low", "close", "upmove", "downmove"]]


In [ ]:
# plot the first 3 reversals. for each reversal plot 20 candles before and 20 after the reversal
for i in range(10):
    start = entries.index[i] - pd.Timedelta(minutes=100)
    end = entries.index[i] + pd.Timedelta(minutes=100)
    fig = go.Figure()
    text = ["\n".join([
        f"CReturns: {round(row['consec_returns']*1e5)}",
        f"upstreak: {row['upcount']}",
        f"downstreak: {row['downcount']}",
        f"idx: {row['indices']}",
    ]) for i, row in df.loc[start:end].iterrows()]
    upmove = entries["upmove"].iloc[i]
    downmove = entries["downmove"].iloc[i]

    fig.add_trace(go.Candlestick(x=df.loc[start:end].index,
                                 open=df.loc[start:end]["open"],
                                 high=df.loc[start:end]["high"],
                                 low=df.loc[start:end]["low"],
                                 close=df.loc[start:end]["close"],
                                 customdata=df[["consec_returns"]].values,
                                 text=text
))
    fig.update_layout(title=f"Reversal {entries.index[i]} / upmove {upmove} downmove {downmove} / Retracement {downmove/upmove}", xaxis_title="Time", yaxis_title="Price")
    # annotate candles with the stage
    for j in range(len(df.loc[start:end])):
        fig.add_annotation(x=df.loc[start:end].index[j],
                            y=df.loc[start:end]["high"].iloc[j],
                            text=f"{df.loc[start:end].iloc[j]['state']}",
                            ax=0,
                            ay=-10,)
    fig.show()


In [ ]:
price =  df.resample('1D').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last'
}).close
price = price.tz_localize(None)  # Remove timezone
price = price.dropna()
price

In [ ]:
kama = ta.kama(price, length=1, fast=2, slow=30)
print(kama)

In [ ]:
price = price.values
kama = kama.values

entries = price > kama
exits = price < kama

pf = vbt.Portfolio.from_signals(price, entries, exits)
fig = pf.plot()

fig.add_trace(go.Scatter(
    y=kama,
    mode='lines',
    name='KAMA',
    line=dict(color='orange')
))
fig.show()

# SMA Crossover

In [ ]:
# Intraday MA windows (scalping & day trading)
intraday_wnds = [5, 10, 15, 30, 60, 90, 120, 180, 240, 360]

# Swing trading MA windows (multi-day)
swing_wnds = [720, 1440, 2880, 4320, 7200, 10080]

# Long-term trading MA windows (weeks+)
long_term_wnds = [20160, 43200, 86400]

# Combine into one list
ma_wnds = intraday_wnds + swing_wnds + long_term_wnds

import itertools

# Generate valid (fast, slow) pairs
valid_pairs = [(fast, slow) for fast, slow in itertools.product(ma_wnds, repeat=2) if fast < slow]

# Separate into lists
fast_ma_wnds, slow_ma_wnds = zip(*valid_pairs)
fast_ma_wnds, slow_ma_wnds = list(fast_ma_wnds), list(slow_ma_wnds)

# Check the first few pairs
len(valid_pairs)


In [ ]:
mas = vbt.MA.run(price, list(set(fast_ma_wnds) | set(slow_ma_wnds))).ma

In [ ]:
def custom_indicator(close, fast_ma_wnd, slow_ma_wnd):
    print(fast_ma_wnd, slow_ma_wnd)
    fast_ma = mas[fast_ma_wnd]
    slow_ma = mas[slow_ma_wnd]

    trend = np.zeros_like(fast_ma)  # Preallocate trend array
    trend[np.where(fast_ma > slow_ma)] = 1
    trend[np.where(fast_ma < slow_ma)] = -1
    return trend


ind = vbt.IndicatorFactory(
    input_names=['close'],
    param_names=['fast_ma_wnd', 'slow_ma_wnd'],
    output_names=['value']
).from_apply_func(custom_indicator)

In [ ]:
res = ind.run(price, fast_ma_wnds, slow_ma_wnds)

In [ ]:
entries = res.value == 1.0
exits = res.value == -1.0

In [ ]:
portfolio = vbt.Portfolio.from_signals(price, entries, exits, freq="1min")

In [ ]:
portfolio[90, 240].stats()

In [ ]:
portfolio.trades.count().unstack()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get total returns for each parameter combination
returns = portfolio.total_return().unstack()

# Create heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(returns, annot=True, cmap="coolwarm", fmt=".2f", annot_kws={"size": 6})
plt.xlabel("Slow MA Window")
plt.ylabel("Fast MA Window")
plt.title("Total Return Heatmap")
plt.show()

# Get total returns for each parameter combination
returns = portfolio.trades.win_rate().unstack()

# Create heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(returns, annot=True, cmap="coolwarm", fmt=".2f", annot_kws={"size": 6})
plt.xlabel("Slow MA Window")
plt.ylabel("Fast MA Window")
plt.title("Win Rate Heatmap")
plt.show()

# Get total returns for each parameter combination
returns = portfolio.trades.count().unstack()

# Create heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(returns, annot=True, cmap="coolwarm", fmt=".0f", annot_kws={"size": 6})
plt.xlabel("Slow MA Window")
plt.ylabel("Fast MA Window")
plt.title("Trade Count Heatmap")
plt.show()


# Find Peaks

In [ ]:
# find peaks
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.signal import find_peaks


In [ ]:
x = np.array(range(len(price)))
y = np.array(price)

# calculate peaks
prom = 0.005
tops, properties = find_peaks(y, prominence=prom)
bottoms, properties = find_peaks(-y, prominence=prom)

# merge tops and bottoms
peaks = np.sort(np.concatenate([tops, bottoms]))

# Plot the price and highlight the peaks using plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.array(range(len(price))), y=price, mode='lines', name='price'))
fig.add_trace(go.Scatter(x=np.array(range(len(price)))[peaks], y=price[peaks], name='peaks', line=dict(width=1)))
fig.show()


In [ ]:
confirmation_points = np.full(peaks.shape, -1)
for i in range(len(peaks)):
    peak = peaks[i]
    for j in range(peaks[i], len(y)):
        if abs(y[j]-y[peak]) >= prom:
            confirmation_points[i] = j
            break

In [ ]:
import plotly.graph_objects as go
window = 1000
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='price'))
fig.add_trace(go.Scatter(x=peaks, y=y[peaks], name='peaks'))
fig.add_trace(go.Scatter(x=confirmation_points, y=y[confirmation_points], mode='markers', name='confirmation points'))
# for i in range(len(peaks)):
#     fig.add_trace(go.Scatter(x=[peaks[i], confirmation_points[i], confirmation_points[i]],
#                              y=[y[peaks[i]], y[peaks[i]], y[confirmation_points[i]]],
#                              mode='lines', line=dict(color='red')))
fig.show()

# Test to understand find_peaks

In [ ]:
# simple find peaks example with 10 points
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

prom = 1.1

# generate data to test prominence
y = np.array([0, 1, 3,2,3,1,1,.5, 0, 2, 0, 3,2,2.5,1.5,2,1, .5, 2, 0, 1])
x = np.arange(len(y))
tops, pt = find_peaks(y, prominence=prom)
bottoms, pb = find_peaks(-y, prominence=prom)

peaks = np.sort(np.concatenate([tops, bottoms]))

plt.plot(x, y, '+-')
plt.plot(x[peaks], y[peaks], "x-")
plt.show()

For tops:
1. rise at least *prominence* units
2. fall at least *prominence* units (this value is the confirmation; not tracked by find_peaks)
3. peak = highest value in that range

**Problem here: First peak is invalid** (equal height)

# Faster Find Peaks
Precompute without prominence to reduce the dataset where it's monotonic

In [ ]:
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

# Generate sample price data
np.random.seed(42)
y = np.cumsum(np.random.randn(20_000_000))  # Simulated price data
x = np.arange(len(y))

prom = 500

In [ ]:
find_peaks(y);

In [ ]:
# benchmark (40s)
find_peaks(y, prominence=prom);

In [ ]:
# precomputed xtop and xbot without prominence (8.6s)
xtop = find_peaks(y)[0]
tops, _ = find_peaks(y[xtop], prominence=prom)

xbot = find_peaks(-y)[0]
bottoms, _ = find_peaks(-y[xbot], prominence=prom)

In [ ]:
peaks = np.sort(np.concatenate([xtop[tops], xbot[bottoms]]))

In [ ]:
plt.plot(y)
plt.plot(peaks, y[peaks])

# Confirmations
Find confirmation points where y retraces from peaks by prominence

In [ ]:
len(peaks)

In [ ]:
confirmation_points = np.full(peaks.shape, -1)
for i in range(len(peaks)):
    peak = peaks[i]
    for j in range(peaks[i], len(y)):
        if abs(y[j]-y[peak]) >= prom:
            confirmation_points[i] = j
            break

confirmation_points
    

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(y, linewidth=0.5)
#plt.plot(peaks, y[peaks])
plt.plot(confirmation_points, y[confirmation_points], "x")
plt.plot(confirmation_points, y[peaks], 'orange')
for i in range(len(peaks)):
    plt.plot(
        [peaks[i], confirmation_points[i], confirmation_points[i]], 
        [y[peaks[i]], y[peaks[i]], y[confirmation_points[i]]], 
        "r")
plt.show()


In [ ]:
# plotly is to slow for so many data points

# same plot with plotly
#import plotly.graph_objects as go
#window = 1000
#fig = go.Figure()
#fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='price'))
#fig.add_trace(go.Scatter(x=peaks, y=y[peaks], name='peaks'))
#fig.add_trace(go.Scatter(x=confirmation_points, y=y[confirmation_points], mode='markers', name='confirmation points'))
#for i in range(len(peaks)):
#    fig.add_trace(go.Scatter(x=[peaks[i], confirmation_points[i], confirmation_points[i]],
#                             y=[y[peaks[i]], y[peaks[i]], y[confirmation_points[i]]],
#                             mode='lines', line=dict(color='red')))
#fig.show()

# Dynamic Support and Resistance

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

IS_TICK = True if 'bid' in df.columns else False

if IS_TICK:
    HIGH_COLNAME = 'bid'
    LOW_COLNAME = 'bid'
    COPEN = 'bid'
    CHIGH = 'bid'
    CLOW = 'bid'
    CCLOSE = 'bid'
else:
    HIGH_COLNAME = 'high'
    LOW_COLNAME = 'low'
    COPEN = 'open'
    CHIGH = 'high'
    CLOW = 'low'
    CCLOSE = 'close'
    

# Parameters
prominence = 0.001      # for peak detection
tolerance = prominence         # price proximity to count as test
break_margin = 0     # margin to declare a level as broken


df_candles = df.resample('1h').agg(
    open = (COPEN, 'first'),
    high = (CHIGH, 'max'),
    low = (CLOW, 'min'),
    close = (CCLOSE, 'last')
)

# Add weekhour for plot truncation
df_candles['weekhour'] = df_candles.index.hour + df_candles.index.weekday * 24

# Broadcast close price 
hourly = df['bid'].resample('1h').last().rename('1h_close')
df = df.drop(columns=['1h_close'], errors='ignore')  # safe to rerun
df['1h_close'] = pd.merge_asof(df, hourly.shift(), left_index=True, right_index=True, direction='backward')['1h_close']


# Detect peaks from unresampled data
highs = df[HIGH_COLNAME].values
lows = df[LOW_COLNAME].values

levels = dict()
for ilt in range(2):
    level_type  = ('Support', 'Resistance')[ilt]
    values = (lows, highs)[ilt]
    sign = (-1, +1)[ilt]

    indices, _ = find_peaks(sign*values, prominence=prominence)
    levels[level_type] = pd.DataFrame(indices, columns=['idx'])
    levels[level_type]['price'] = values[indices]
    levels[level_type]['type'] = level_type
    levels[level_type]['created_at'] = df.index[indices]

    #closes = [df["close"].iloc[idx:] for idx in indices]
    #thresholds = values[indices] * (1 + sign * break_margin)
    #breaks = sign * closes > sign * thresholds
    levels[level_type]['invalidated_at'] = [
        (sign * df_candles[df_candles.index > level['created_at']].open > sign * level["price"] * (1 + sign * break_margin)).idxmax() 
        if (sign * df_candles[df_candles.index > level['created_at']].open > sign * level["price"] * (1 + sign * break_margin)).any() 
        else None
        for i, level in levels[level_type].iterrows()]

levels = pd.concat(levels.values())
levels.sort_values(by='idx', inplace=True)

# make the levels alternating, remove consecutive levels of the same type (price will be the same), keep the first one
levels['level_id'] = (levels['type'] != levels['type'].shift()).cumsum()
levels = levels.groupby('level_id', as_index=False).first().drop(columns='level_id')

levels

# Zones from S/R

In [ ]:
class Level:
    def __init__(self, level: pd.Series): #type: str, price: float, created_at: pd.Timestamp):
        self.type = level['type']
        self.price = level['price']
        self.created_at = level['created_at']
        self.invalidated_at = level['invalidated_at']
        self.zone = None  # will be set by Zone.add_level()
    
    # def invalidate(self, invalidated_at: pd.Timestamp):
    #     self.invalidated_at = invalidated_at
    #     self.zone.remove_level(self)
    

class Zone:
    def __init__(self, level):
        self.id = None  # will be set by Zones.add_zone()
        self.type = level.type
        self.created_at = level.created_at
        self.invalidated_at = None
        self.levels = [level]
        self.timestamps = [level.created_at]
        self.min_prices = [level.price]
        self.max_prices = [level.price]
        self.mean_prices = [level.price]

        level.zone = self
    
    def __repr__(self):
        return f"Zone(type={self.type}, created_at={self.created_at}, invalidated_at={self.invalidated_at}, levels={len(self.levels)})"

    def add_level(self, level):
        self.levels.append(level)
        self.timestamps.append(level.created_at)
        self.update_prices(level.created_at)
        level.zone = self

    def remove_level(self, level):
        self.timestamps.append(level.invalidated_at)
        self.update_prices(level.invalidated_at)

    def update_prices(self, timestamp):
        active_prices = self.active_prices(timestamp)
        if active_prices:
            self.min_prices.append(np.min(active_prices))
            self.max_prices.append(np.max(active_prices))
            self.mean_prices.append(np.mean(active_prices))
        else:
            self.min_prices.append(self.min_prices[-1])
            self.max_prices.append(self.max_prices[-1])
            self.mean_prices.append(self.mean_prices[-1])
            self.invalidated_at = timestamp
    
    def active_prices(self, timestamp):
        return [l.price for l in self.levels if (l.invalidated_at is None) or (l.invalidated_at > timestamp)]
    
class Zones(list):

    def add_zone(self, zone: Zone):
        self.append(zone)
        zone.id = len(self) - 1


    def find_zone(self, price: float, zone_type: str, tolerance: float = 0.001):
        """Find a zone that contains the price within the tolerance."""
        for zone in self:
            if zone.invalidated_at is not None:
                continue
            if zone.type != zone_type:
                continue
            if abs(zone.mean_prices[-1] - price) <= tolerance:
                return zone
        return None

In [ ]:
# track sorted timestamps when levels were created and invalidated
timestamps = sorted([*levels.created_at, *levels.invalidated_at.dropna().unique()])

# add Level objecs to the levels DataFrame
levels['object'] = levels.apply(lambda x: Level(x), axis=1)

# Initialize zones container
zones = Zones()

for t in timestamps:
    # Filter levels that were created or invalidated at this timestamp
    created_levels = levels[levels['created_at'] == t]['object']
    invalidated_levels = levels[levels['invalidated_at'] == t]['object']

    # Add created levels or create new zones
    for level in created_levels:
        nearby_zone = zones.find_zone(level.price, level.type, tolerance)
        if nearby_zone is None:
            zones.add_zone(Zone(level))
        else:
            nearby_zone.add_level(level)
    
    # Remove invalidated levels from their zones
    for level in invalidated_levels:
        level.zone.remove_level(level)

In [ ]:
zones[0].timestamps, zones[0].min_prices, zones[0].max_prices, zones[0].mean_prices

In [ ]:
# Track active zones by type
zones = {
    'Support': [],
    'Resistance': []
}

for idx, row in levels.iterrows():
    level_type = row['type']
    price = row['price']

    assigned_zone = None

    for zone in zones[level_type]:
        price_within_tolerance = (abs(zone['mean_price'][-1] - price) <= tolerance)
        zone_not_invalidated = (zone['invalidated_at'] is None or row['created_at'] <= zone['invalidated_at'])

        if price_within_tolerance and zone_not_invalidated:
            assigned_zone = zone
            break
    
    if assigned_zone is None:
        # Create a new zone
        zone_id = len(zones[level_type]) + 1
        assigned_zone = {
            'zone_id': zone_id,
            'type': level_type,
            'datetime': [row['created_at']],
            'mean_price': [price],
            'min_price': [price - tolerance],
            'max_price': [price + tolerance],
            'created_at': row['created_at'],
            'invalidated_at': row['invalidated_at'],
            'level_indices': [idx],
        }
        zones[level_type].append(assigned_zone)
    else:
        # Update mean_price, invalidated_at, and level_indices
        zone_id = assigned_zone['zone_id']
        assigned_zone['level_indices'].append(idx)

        zone_levels = levels.loc[zone['level_indices'], :]
        active_zone_levels = zone_levels[zone_levels['invalidated_at'] > row['created_at']]
        zone_prices = active_zone_levels['price'].values

        count = len(assigned_zone['level_indices'])
        assigned_zone['datetime'].append(row['created_at'])
        assigned_zone['mean_price'].append(zone_prices.mean())
        assigned_zone['min_price'].append(zone_prices.min() - tolerance)
        assigned_zone['max_price'].append(zone_prices.max() + tolerance)

        # compare invalidated_at (timestamps) and keep the latest one
        if assigned_zone['invalidated_at'] is None or (row['invalidated_at'] is not None and row['invalidated_at'] > assigned_zone['invalidated_at']):
            assigned_zone['invalidated_at'] = row['invalidated_at']


# Plot Support and Resistance (initial plot)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(layout=go.Layout(width=1200, height=800))

# ticks
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['bid'],
    name='Ticks'
))

# # ticks
# fig.add_trace(go.Scatter(
#     x=df.index,
#     y=df['1h_close'],
#     name='1h_close'
# ))

# candles
fig.add_trace(go.Candlestick(x=df_candles.index + pd.Timedelta(hours=0.5),
                             open=df_candles["open"],
                             high=df_candles["high"],
                             low=df_candles["low"],
                             close=df_candles["close"],
                             name='Candlestick',
                             hovertext=df_candles['weekhour']
                             ))

# zones
# polygon for the zone with stepped min/max prices
for i, zone in enumerate(zones):
        # Create a polygon for the zone
        pol_x = zone.timestamps
        pol_x += [zone.invalidated_at] if zone.invalidated_at is not None else [df_candles.index[-1]]
        pol_x += [pol_x[-1]] 
        pol_x += zone.timestamps[::-1]

        fig.add_trace(go.Scatter(
            x=pol_x,
            y=zone.min_prices + [zone.min_prices[-1], zone.max_prices[-1]] + zone.max_prices[::-1],
            fill='toself',
            fillcolor=('rgba(255, 0, 0, 0.2)', 'rgba(0, 255, 0, 0.2)')[('Support', 'Resistance').index(zone.type)],
            line=dict(color='rgba(0, 0, 0, 0)'),
            name=f"{zone.type} Zone {zone.id}",
            legendgroup=f"{zone.type} Zone",
            showlegend=True if i<2 else False,
        ))
        # zone mean
        fig.add_trace(go.Scatter(
            x=zone.timestamps,
            y=zone.mean_prices,
            mode='lines',
            line=dict(color=('red', 'green')[('Support', 'Resistance').index(zone.type)], width=2),
            name=f"{zone.type} Mean",
            legendgroup=f"{zone.type} Mean",
            line_dash='dash',
            showlegend=True if i<2 else False,
        ))

# levels
show_legend = [True, True]
for i, peak in levels.iterrows():
    tfrom = peak["created_at"]
    to = peak["invalidated_at"] if peak["invalidated_at"] is not pd.NaT else df.index[-1]
    fig.add_trace(go.Scatter(
        x=[tfrom, to], 
        y=[peak.price, peak.price],
        mode='lines',
        line_color=('darkred', 'darkgreen')[('Support', 'Resistance').index(peak.type)],
        name=peak.type,
        legendgroup=peak.type,
        showlegend=True if i < 2 else False,
        # annotate with id only
        text=[f"ID: {i}"],
        ))

# zigzag
fig.add_trace(go.Scatter(
    x=levels['created_at'], 
    y=levels['price'],
    name='Zigzag',
    line_color='black',
    ))


fig.update_xaxes(
    # hide weekend from fr 10pm to sunday 10pm (forex market)
    rangebreaks=[
        dict(values=df_candles.query("weekhour > 119 and weekhour < 165").index),  # hide weekends
    ],
    # daily ticks
    dtick='D1',
)

fig.update_layout(
    title='Candlestick Chart with Support/Resistance Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    yaxis=dict(fixedrange=False),  # Allow zooming in the y-direction
    #no range slider
    xaxis_rangeslider_visible=False,
)

#fig.show()


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(layout=go.Layout(width=1200, height=800))

# # ticks
# fig.add_trace(go.Scatter(
#     x=df.index,
#     y=df['bid'],
#     name='Ticks'
# ))

# # ticks
# fig.add_trace(go.Scatter(
#     x=df.index,
#     y=df['1h_close'],
#     name='1h_close'
# ))

# candles
fig.add_trace(go.Candlestick(x=df_candles.index + pd.Timedelta(hours=0.5),
                             open=df_candles["open"],
                             high=df_candles["high"],
                             low=df_candles["low"],
                             close=df_candles["close"],
                             name='Candlestick',
                             hovertext=df_candles['weekhour']
                             ))

# levels
show_legend = [True, True]
for i, peak in levels.iterrows():
    tfrom = peak["created_at"]
    to = peak["invalidated_at"] if peak["invalidated_at"] is not pd.NaT else df.index[-1]
    fig.add_trace(go.Scatter(
        x=[tfrom, to], 
        y=[peak.price, peak.price],
        mode='lines',
        line_color=('darkred', 'darkgreen')[('Support', 'Resistance').index(peak.type)],
        name=peak.type,
        legendgroup=peak.type,
        showlegend=show_legend[('Support', 'Resistance').index(peak.type)],
        ))
    show_legend[('Support', 'Resistance').index(peak.type)] = False  # only show legend for first peak of each type

# zones
# polygon for the zone with stepped min/max prices
for zone_type, zone_list in zones.items():
    for zone in zone_list:
        if len(zone['mean_price']) < 2:
            continue  # Skip zones with insufficient data

        # Create a polygon for the zone
        fig.add_trace(go.Scatter(
            x=zone['datetime'] + zone['datetime'][::-1],
            y=zone['min_price'] + zone['max_price'][::-1],
            fill='toself',
            fillcolor=('rgba(255, 0, 0, 0.2)', 'rgba(0, 255, 0, 0.2)')[('Support', 'Resistance').index(zone_type)],
            line=dict(color='rgba(0, 0, 0, 0)'),
            name=f"{zone_type} Zone {zone['zone_id']}",
            legendgroup=zone_type,
            showlegend=True if zone['zone_id'] <= 2 else False,
        ))
    

# zigzag
fig.add_trace(go.Scatter(
    x=levels['created_at'], 
    y=levels['price'],
    name='Zigzag',
    line_color='black',
    ))


fig.update_xaxes(
    # hide weekend from fr 10pm to sunday 10pm (forex market)
    rangebreaks=[
        dict(values=df_candles.query("weekhour > 119 and weekhour < 165").index),  # hide weekends
    ],
    # daily ticks
    dtick='D1',
)

fig.update_layout(
    title='Candlestick Chart with Support/Resistance Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    yaxis=dict(fixedrange=False),  # Allow zooming in the y-direction
    #no range slider
    xaxis_rangeslider_visible=False,
)

#fig.show()


# Plot Trading Sessions (updates plot)

In [ ]:
from trader.indicators import trading_session as ts

sessions = ts.get_sessions(df)
fig = ts.plot_sessions(sessions, fig=fig)
fig.show()

# Get News Events

In [ ]:
from trader.indicators import economic_data
from datetime import datetime, timezone, timedelta

date_slices = pd.date_range(
    df.index[0].date(), 
    df.index[-1].date(), 
    freq='1MS'
)
date_slices = [df.index[0].date(), *date_slices.to_list(), df.index[-1].date()]
date_slices = [date_slices[i:i + 2] for i in range(len(date_slices) - 1)]

df_news = pd.concat([economic_data.get_economic_data(start_date.isoformat(), end_date.isoformat(), utc=True) for start_date, end_date in date_slices])
df_news = df_news.query("currency == 'USD'")

# make datetimeindex tz aware
#df_news.index = df_news.index.tz_localize(timezone(datetime.utcnow() - datetime.now())).tz_convert("UTC")
df_news.index = df_news.index.tz_localize("UTC")
df_news[df_news.index.date == datetime.fromisoformat("2024-06-07").date()]

# Plot News Events (updates plot)

In [ ]:
import plotly.graph_objects as go

DEFAULT_NAME = 'NewsEvent'
LEGEND_OPTIONS = ['currency', 'impact', 'name']
LEGEND_BY = LEGEND_OPTIONS[2]  # Change this to select legend grouping

# Sort by index (datetime)
df_news = df_news.sort_index()

# Assign a vertical stacking position for repeated timestamps
df_news['y'] = 0
height_counter = {}

for i, ts in enumerate(df_news.index):
    height_counter[ts] = height_counter.get(ts, 0) + 1
    df_news.iat[i, df_news.columns.get_loc('y')] = height_counter[ts]

# Map impact to colors
impact_colors = {1: 'gray', 2: 'orange', 3: 'red'}
df_news['color'] = df_news['impact'].map(impact_colors).fillna('blue')

# Create Plotly figure
#fig = go.Figure()

shown_legend_groups = set()
for timestamp, row in df_news.iterrows():
    last_candle_time = df_candles.index[df_candles.index.get_indexer([timestamp], method='bfill')][0]
    legend_group = row[LEGEND_BY] if LEGEND_BY else DEFAULT_NAME
    show_legend = legend_group not in shown_legend_groups
    if show_legend:
        shown_legend_groups.add(legend_group)

    fig.add_trace(go.Scatter(
        x=[timestamp],
        y=[df_candles.loc[last_candle_time, 'low'] - 0.001 * row['y']],  # Adjust y position based on stacking
        mode='markers+text',
        marker=dict(color=row['color'], size=10),
        name=legend_group,
        legendgroup=legend_group,
        hovertemplate=(
            f"Time: {timestamp}<br>"
            f"Currency: {row['currency']}<br>"
            f"Name: {row['name']}<br>"
            f"Impact: {row['impact']}<extra></extra><br>"
            f"Forecast: {row['forecast']}<br>"
            f"Actual: {row['actual']}<br>"
            f"Previous: {row['previous']}<br>"
        ),
        showlegend=show_legend
    ))

# Layout configuration
# fig.update_layout(
#     title="Economic Events Timeline (Stacked by Timestamp)",
#     xaxis_title="Time",
#     yaxis_title="Stacked Events",
#     yaxis=dict(showticklabels=False),
#     template="plotly_white",
#     height=500
# )

fig.show()
#save to html
fig.write_html("eurusd_june_24.html")


# fig -> Dash

In [ ]:
from dash import dcc, html, Dash, Input, Output, State, callback_context
import plotly.express as px
import plotly.graph_objects as go

app = Dash()

# Load data and initial figure
df = px.data.iris()
base_fig = fig#px.scatter(df, x="sepal_width", y="sepal_length")

app.layout = html.Div([
    html.Div([
        html.Button("Free Zoom", id="btn-free", n_clicks=0),
        html.Button("X-only Zoom", id="btn-x", n_clicks=0),
        html.Button("Y-only Zoom", id="btn-y", n_clicks=0),
    ], style={"marginBottom": "10px"}),

    dcc.Graph(id="graph", config={"scrollZoom": True})
])

@app.callback(
    Output("graph", "figure"),
    Input("btn-free", "n_clicks"),
    Input("btn-x", "n_clicks"),
    Input("btn-y", "n_clicks"),
    State("graph", "figure")
)
def update_zoom_mode(n_free, n_x, n_y, fig_dict):
    # Identify which button was clicked
    ctx = callback_context
    triggered_id = ctx.triggered[0]["prop_id"].split(".")[0] if ctx.triggered else "btn-free"

    fig = go.Figure(fig_dict) if fig_dict else base_fig

    # Set zoom mode based on button clicked
    if triggered_id == "btn-x":
        fig.update_layout(yaxis_fixedrange=True, xaxis_fixedrange=False)
    elif triggered_id == "btn-y":
        fig.update_layout(xaxis_fixedrange=True, yaxis_fixedrange=False)
    else:
        fig.update_layout(xaxis_fixedrange=False, yaxis_fixedrange=False)
    # Else allow free zoom (do not fix any axis)

    # Always allow scroll zoom
    fig.update_layout(dragmode="zoom")

    return fig

if __name__ == '__main__':
    app.run(debug=True)


# Kernel Density Estimation

In [ ]:
# slice the dataframe to last week
df_last_week = df.last("7D")
df_last_week

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

values = df_last_week['bid'].values

# Kernel Density Estimation
kde = gaussian_kde(values)
value_range = np.linspace(min(values), max(values), 200)
density = kde(value_range)

# tick volume
tick_volume_1h = df_last_week.resample('1h').size()
tick_volume_1d = df_last_week.resample('1D').size()

# Calculate bar widths
hour_width = 1 * tick_volume_1h.index.diff().total_seconds().to_series().median() / (60*60*24)
day_width = 1 * tick_volume_1d.index.diff().total_seconds().to_series().median() / (60*60*24)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax2 = ax.inset_axes([1.05, 0, 0.2, 1], sharey=ax)  # KDE on the right
ax3 = ax.inset_axes([0, -0.3, 1, 0.2], sharex=ax)  # Tick volume 
ax4 = ax.inset_axes([0, -0.6, 1, 0.2], sharex=ax)  # Tick volume

# Plot time series
ax.plot(df_last_week.index, values, label='Time Series')
ax.set_xlabel('Time')
ax.set_ylabel('Bid Price')
ax.set_title('Time Series with KDE')

# Plot rotated KDE
ax2.plot(density, value_range)
ax2.set_xlabel('Density')
ax2.set_yticks([])
ax2.set_xticks([])
ax2.set_title('KDE', fontsize=10)

# Plot tick volume
ax3.bar(tick_volume_1h.index, tick_volume_1h.values, width=hour_width, color='orange', align='edge', edgecolor='black', label='Tick Volume')
ax3.set_yticks([])
ax3.set_ylabel('Tick Volume')
ax4.bar(tick_volume_1d.index, tick_volume_1d.values, width=day_width, color='orange', align='edge', edgecolor='black', label='Tick Volume')
ax4.set_yticks([])
ax4.set_ylabel('Tick Volume')

plt.show()

In [ ]:
0.8 * tick_volume_1h.index.diff().total_seconds().to_series().median() 
